In [2]:
import bioprov as bp
from bioprov.programs import prodigal

In [ ]:
proj = bp.load_project("bp-use-case")

for k, sample in proj.items():
    p = prodigal(input_tag="genome_assembly")
    del p.output_files["-s"]
    p.create_func(proj["GCA_003990665.2"])
    sample.add_programs(p)
    sample.run_programs()

Updating file proteins with value /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic_proteins.faa.
Updating file genes with value /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic_genes.fna.
Updating file proteins with value /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic_proteins.faa.
Updating file genes with value /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic_genes.fna.
Running program 'prodigal' for sample GCA_003990665.2.
Command is:
/Users/vini/anaconda3/envs/bp-use-case/bin/prodigal \ 
	-i /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic.fna \ 
	-a /Users/vini/Bio/BioProv/use-case/data/genbank/bacteria/GCA_003990665.2/GCA_003990665.2_ASM399066v2_genomic_proteins.faa \ 
	-d /Users/vini/Bio/BioPr

In [84]:
proj = bp.load_project("bp-use-case")

In [85]:
fastani_input = bp.File("data/fastani_input.txt")
fastani_output = bp.File("data/fastani_output.txt")
proj.add_files([fastani_input, fastani_output])

with open(proj.files["fastani_input"].path, "w") as f:
    for file in (sample.files["genome_assembly"] for k, sample in proj.items()):
        f.write(str(file) + "\n")

Updating file fastani_input with value /Users/vini/Bio/BioProv/use-case/data/fastani_input.txt.
Updating file fastani_output with value /Users/vini/Bio/BioProv/use-case/data/fastani_output.txt.


In [86]:
fastani = bp.Program("fastani")

params = [
    bp.Parameter("--refList", proj.files["fastani_input"], kind="input"),
    bp.Parameter("--queryList", proj.files["fastani_input"]),
    bp.Parameter("--threads", bp.config.threads),
    bp.Parameter("--fragLen", 200),
    bp.Parameter("--minFraction", 0.01),
    bp.Parameter("--output", proj.files["fastani_output"], kind="output")
]

for param in params:
    fastani.add_parameter(param)

In [87]:
proj.add_programs(fastani)
fastani.run()

Running program 'fastani'.
Command is:
/Users/vini/anaconda3/envs/bp-use-case/bin/fastani \ 
	--refList /Users/vini/Bio/BioProv/use-case/data/fastani_input.txt \ 
	--queryList /Users/vini/Bio/BioProv/use-case/data/fastani_input.txt \ 
	--threads 2 \ 
	--fragLen 200 \ 
	--minFraction 0.01 \ 
	--output /Users/vini/Bio/BioProv/use-case/data/fastani_output.txt 


Run of Program 'fastani' with 6 parameter(s).
Started at Tue Jan  5 14:44:37 2021.
Ended at Tue Jan  5 14:52:15 2021.
Status is finished.

In [88]:
proj.update_db()

In [89]:
# ffo stands for 'fastani output formatted'
ffo_f = bp.File("data/fastani_output_fmt.txt")
proj.add_files(ffo_f)

ffo_p = bp.Program("format_fastani_output")
ffo_p.found = True

params = [
    bp.Parameter("python", "format_fastani_output.py"),
    bp.Parameter("-i", proj.files["fastani_output"], kind="input"),
    bp.Parameter("-o", proj.files["fastani_output_fmt"], kind="output"),
    bp.Parameter("--invert")
]

for param in params:
    ffo_p.add_parameter(param)
    
proj.add_programs(ffo_p)
ffo_p.run()

Updating file fastani_output_fmt with value /Users/vini/Bio/BioProv/use-case/data/fastani_output_fmt.txt.
Running program 'format_fastani_output'.
Command is:
python \ 
	format_fastani_output.py -i \ 
	/Users/vini/Bio/BioProv/use-case/data/fastani_output.txt -o \ 
	/Users/vini/Bio/BioProv/use-case/data/fastani_output_fmt.txt --invert 


Run of Program 'format_fastani_output' with 4 parameter(s).
Started at Tue Jan  5 14:52:15 2021.
Ended at Tue Jan  5 14:52:16 2021.
Status is finished.

In [90]:
print(ffo_p.runs['1'].stderr)

In [91]:
proj.update_db()

In [35]:
prov = bp.BioProvDocument(proj)

In [36]:
prov.dot.write_pdf("graphs/processing_2.pdf")

In [33]:
# Create labels file
proj = bp.load_project("bp-use-case")

In [69]:
with open("data/labels.txt", "w") as f:
    for k, sample in proj.items():
        filename = str(sample.files["genome_assembly"].basename)
        label = sample.attributes["organism_name"]
        f.write(f"{filename},{label}\n")

In [92]:
labels = bp.File("data/labels.txt")
dendrogram = bp.File("data/dendrogram.pdf")
proj.add_files([labels, dendrogram])

clust = bp.Program("cluster_and_plot")
clust.found = True

params = [
    bp.Parameter("python", "hierarchical_clustering.py"),
    bp.Parameter("-t", proj.files["fastani_output_fmt"], kind="input"),
    bp.Parameter("-l", proj.files["labels"], kind="misc"),
    bp.Parameter("-o", proj.files["dendrogram"], kind="output"),
    bp.Parameter("-c", 22.5, kind="misc")
]

for param in params:
    clust.add_parameter(param)
    
proj.add_programs(clust)
clust.run()

Updating file labels with value /Users/vini/Bio/BioProv/use-case/data/labels.txt.
Updating file dendrogram with value /Users/vini/Bio/BioProv/use-case/data/dendrogram.pdf.
Running program 'cluster_and_plot'.
Command is:
python \ 
	hierarchical_clustering.py -t \ 
	/Users/vini/Bio/BioProv/use-case/data/fastani_output_fmt.txt -l \ 
	/Users/vini/Bio/BioProv/use-case/data/labels.txt -o \ 
	/Users/vini/Bio/BioProv/use-case/data/dendrogram.pdf -c \ 
	22.5


Run of Program 'cluster_and_plot' with 5 parameter(s).
Started at Tue Jan  5 14:52:17 2021.
Ended at Tue Jan  5 14:52:19 2021.
Status is finished.

In [93]:
proj.update_db()

In [94]:
prov = bp.BioProvDocument(proj)
prov.dot.write_pdf("graphs/processing_3.pdf")